# A system for detecting the **malicious activities in the network traffic**

In [13]:
import pandas as pd
import hashlib
import numpy as np

## Uploading the Dataset

In [14]:
# Load dataset
df = pd.read_csv("/kaggle/input/network-traffic-dataset/Midterm_53_group.csv")
df.head()

,Time,Source,No.,Destination,Protocol,Length,Info
0,0.000000,192.167.8.166,1,192.167.255.255,NBNS,92,Name query NB WPAD<00>
1,0.784682,192.167.8.166,2,192.167.255.255,NBNS,92,Name query NB WPAD<00>
2,1.169060,VMware_8a:5c:e6,3,Broadcast,ARP,60,Who has 192.167.7.175? Tell 192.167.0.1
3,2.167949,VMware_8a:5c:e6,4,Broadcast,ARP,60,Who has 192.167.7.175? Tell 192.167.0.1
4,3.170095,VMware_8a:5c:e6,5,Broadcast,ARP,60,Who has 192.167.7.175? Tell 192.167.0.1


## Taking out only `Source` and `Destination` columns

In [15]:
# Extract Source and Destination IP
df_filtered = df[['Source', 'Destination']]

print(df_filtered.head())

            Source      Destination
0    192.167.8.166  192.167.255.255
1    192.167.8.166  192.167.255.255
2  VMware_8a:5c:e6        Broadcast
3  VMware_8a:5c:e6        Broadcast
4  VMware_8a:5c:e6        Broadcast


## Helper functions

### Count Trailing Zeros

In [16]:
# --- Helper Function: Count Trailing Zeros ---
def count_trailing_zeros(x):
    """Count trailing zeros in the binary representation of x."""
    if x == 0:
        return 0
    # x & -x isolates the lowest set bit; its bit length minus one gives the count of trailing zeros.
    return (x & -x).bit_length() - 1

## FM Algorithm function

In [17]:
# --- FM Algorithm Function ---
def fm_estimate(stream, num_hashes=10):
    """
    Estimate the number of distinct elements in the stream using the FM algorithm.
    
    Parameters:
      stream: List of IP addresses (as strings).
      num_hashes: Number of hash functions (different seeds) to use.
    
    Returns:
      The FM estimate averaged over the hash functions.
    """
    estimates = []
    for seed in range(num_hashes):
        max_zeros = 0
        for ip in stream:
            # Use MD5 with an extra seed string to simulate different hash functions.
            hash_value = int(hashlib.md5((ip + str(seed)).encode('utf-8')).hexdigest(), 16)
            tz = count_trailing_zeros(hash_value)
            if tz > max_zeros:
                max_zeros = tz
        # Correction factor phi for the FM algorithm (approximately 0.77351)
        phi = 0.77351
        estimate = (2 ** max_zeros) / phi
        estimates.append(estimate)
    return np.mean(estimates)

## Sliding Window Emulator

In [18]:
# --- Sliding Window Analysis ---
def sliding_window_fm(df, window_size, step, num_hashes=10):
    """
    Apply the FM algorithm over a sliding window on the source IP stream.
    
    Parameters:
      df: DataFrame with at least a 'Source' column.
      window_size: Number of rows (or packets) in each window.
      step: Step size for sliding window.
      num_hashes: Number of hash functions for the FM estimate.
      
    Returns:
      A DataFrame with FM estimates, actual distinct counts, and error per window.
    """
    results = []
    for start in range(0, len(df) - window_size + 1, step):
        window = df.iloc[start:start+window_size]
        source_ips = window['Source'].tolist()
        fm_est = fm_estimate(source_ips, num_hashes)
        actual_distinct = len(set(source_ips))
        error = abs(fm_est - actual_distinct) / actual_distinct if actual_distinct > 0 else 0
        results.append({
            'window_start': start,
            'window_end': start + window_size,
            'fm_estimate': fm_est,
            'actual_distinct': actual_distinct,
            'relative_error': error
        })
    return pd.DataFrame(results)

## Setting the parameters
- window size (N) = 100
- step (s) = 50 (Sliding window moves 50 packets each time)
- number of hashes (|H|) = 10

In [19]:
# Set parameters for the sliding window.
window_size = 100
step = 50
num_hashes = 10

## Computing the FM estimates over the sliding windows

In [20]:
# Compute the FM estimates over the sliding windows.
results_df = sliding_window_fm(df_filtered, window_size, step, num_hashes)
print("\nSliding window analysis results:")
print(results_df.head())


Sliding window analysis results:
   window_start  window_end  fm_estimate  actual_distinct  relative_error
0             0         100    56.883557               12        3.740296
1            50         150    18.357875                5        2.671575
2           100         200     4.266267                1        3.266267
3           150         250     4.266267                1        3.266267
4           200         300     4.266267                1        3.266267


## Detecting Attacks
### One simple heuristic: flag windows where the FM estimate is significantly higher than a baseline.

Here, we compute a baseline using the median of actual distinct counts.

In [22]:
baseline = results_df['actual_distinct'].median()

### If the FM estimate exceeds 1.5 times the baseline, we mark it as suspicious.

In [23]:
results_df['suspicious'] = results_df['fm_estimate'] > (1.5 * baseline)
print("\nPotentially malicious windows (based on distinct count anomaly):")
print(results_df[results_df['suspicious']])


Potentially malicious windows (based on distinct count anomaly):
      window_start  window_end  fm_estimate  actual_distinct  relative_error  \
0                0         100    56.883557               12        3.740296   
1               50         150    18.357875                5        2.671575   
2              100         200     4.266267                1        3.266267   
3              150         250     4.266267                1        3.266267   
4              200         300     4.266267                1        3.266267   
...            ...         ...          ...              ...             ...   
7874        393700      393800     4.136986                2        1.068493   
7875        393750      393850     4.136986                2        1.068493   
7876        393800      393900     4.136986                2        1.068493   
7877        393850      393950     4.136986                2        1.068493   
7878        393900      394000     4.136986           

## Performance Analysis ---
Evaluate the FM algorithm performance by computing the mean relative error over all windows.


In [24]:
mean_relative_error = results_df['relative_error'].mean()
print(f"\nMean Relative Error of FM estimation: {mean_relative_error:.3f}")


Mean Relative Error of FM estimation: 3.835


## Explanation
**Detection of Malicious Windows**:
Windows where the estimated distinct count significantly exceeds a baseline (for example, 1.5 times the median distinct count) are flagged as suspicious. Such anomalies might indicate attacks like distributed denial-of-service (DDoS) where a surge in source IP addresses is common.

**Performance Metrics**:

- *Accuracy of Detection*: How many windows with true anomalies (or high actual distinct counts) are correctly flagged as suspicious.
- *Accuracy of Distinct Count Estimation*: Measured here by the relative error between the FM estimate and the actual distinct count.

(*extra*)
### If we use AMS algorithm
```python
def ams_estimate(stream, num_estimators=10):
    """
    Estimate the second moment (F2) using the AMS algorithm.
    
    Parameters:
      stream: List of source IP addresses.
      num_estimators: Number of estimators for averaging.
    
    Returns:
      The AMS estimate of the second moment.
    """
    estimates = []
    for _ in range(num_estimators):
        # Randomly select an index from the stream
        random_index = np.random.randint(0, len(stream))
        sampled_ip = stream[random_index]
        # Count frequency of the sampled IP in the stream
        f = stream.count(sampled_ip)
        # For the AMS algorithm, the estimator is (n * f^2) where n is the stream length,
        # appropriately normalized and averaged over multiple estimators.
        estimate = len(stream) * (f ** 2)
        estimates.append(estimate)
    return np.mean(estimates)

# Using sliding window to compute both FM and AMS estimates:
def sliding_window_analysis(df, window_size, step, num_hashes=10, num_estimators=10):
    results = []
    for start in range(0, len(df) - window_size + 1, step):
        window = df.iloc[start:start+window_size]
        source_ips = window['Source'].tolist()
        
        fm_est = fm_estimate(source_ips, num_hashes)
        ams_est = ams_estimate(source_ips, num_estimators)
        actual_distinct = len(set(source_ips))
        
        results.append({
            'window_start': start,
            'window_end': start + window_size,
            'fm_estimate': fm_est,
            'ams_estimate_F2': ams_est,
            'actual_distinct': actual_distinct
        })
    return pd.DataFrame(results)

```